# DSS_View — first consumer-bases view

Produces `consumer-bases/interim/DSS_View.xlsx` with three sheets:

| Sheet | Grain | Source |
|---|---|---|
| `ReadMe` | — | provenance and column dictionary |
| `Test_Identity` | one row per TESTCD | `Variables` TESTCD-pins, aggregated by TESTCD; NCIt enrichment from `AssignedTerms` (graph) and `SDTM_Test_Identity.xlsx` (reference) |
| `Measurement_Specs` | one row per DSS | `DSS` + `BC` + `Coding` joined; lossless pivot of all `Variables` pins with the leading domain code stripped |

Graph-wide scope: every COSMoS DSS across all 32 domains. Observation-class
scoping (Findings vs Events vs Interventions vs Special-Purpose) is applied
by the consumer track. This view is the joined precursor consumed by
`sdtm-findings/` (and future consumer tracks). It does not carry sub-typing,
behavioural classification, or narrative framing — those stay in the consumer.

## Inputs

| File | Track | Sheets used |
|---|---|---|
| `cosmos-graph/interim/COSMoS_Graph.xlsx` | cosmos-graph | DSS, BC, Variables, Coding, BC_Categories |
| `cosmos-graph/interim/COSMoS_Graph_CT.xlsx` | cosmos-graph | AssignedTerms |
| `sdtm-test-codes/machine_actionable/SDTM_Test_Identity.xlsx` | sdtm-test-codes | Test Codes |
| `sdtm-domain-reference/machine_actionable/SDTM_Domain_Metadata.xlsx` | sdtm-domain-reference | Domains |

## Output

`consumer-bases/interim/DSS_View.xlsx`


## 1. Setup


In [1]:
import pandas as pd
from pathlib import Path
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

print('DSS_VIEW — first consumer-bases view')
print(f'Run: {datetime.now():%Y-%m-%d %H:%M}')


DSS_VIEW — first consumer-bases view
Run: 2026-04-25 12:03


In [2]:
BASE_DIR = Path.cwd().parent          # consumer-bases/
REPO_ROOT = BASE_DIR.parent           # cdisc-for-ai/

GRAPH_FILE = REPO_ROOT / 'cosmos-graph' / 'interim' / 'COSMoS_Graph.xlsx'
GRAPH_CT_FILE = REPO_ROOT / 'cosmos-graph' / 'interim' / 'COSMoS_Graph_CT.xlsx'
TEST_IDENTITY_FILE = REPO_ROOT / 'sdtm-test-codes' / 'machine_actionable' / 'SDTM_Test_Identity.xlsx'
DOMAIN_META_FILE = REPO_ROOT / 'sdtm-domain-reference' / 'machine_actionable' / 'SDTM_Domain_Metadata.xlsx'

INTERIM_DIR = BASE_DIR / 'interim'
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = INTERIM_DIR / 'DSS_View.xlsx'

for f, label in [
    (GRAPH_FILE, 'Graph'),
    (GRAPH_CT_FILE, 'Graph CT'),
    (TEST_IDENTITY_FILE, 'Test Identity'),
    (DOMAIN_META_FILE, 'Domain Meta'),
]:
    if f.exists():
        print(f'  {label}: {f.relative_to(REPO_ROOT)}')
    else:
        raise FileNotFoundError(f'{label} file not found: {f}')

print(f'  Output: {OUTPUT_FILE.relative_to(REPO_ROOT)}')


  Graph: cosmos-graph/interim/COSMoS_Graph.xlsx
  Graph CT: cosmos-graph/interim/COSMoS_Graph_CT.xlsx
  Test Identity: sdtm-test-codes/machine_actionable/SDTM_Test_Identity.xlsx
  Domain Meta: sdtm-domain-reference/machine_actionable/SDTM_Domain_Metadata.xlsx
  Output: consumer-bases/interim/DSS_View.xlsx


## 2. Load graph and reference inputs


In [3]:
dss_g = pd.read_excel(GRAPH_FILE, sheet_name='DSS', dtype=str).fillna('')
bc_g = pd.read_excel(GRAPH_FILE, sheet_name='BC', dtype=str).fillna('')
vars_g = pd.read_excel(GRAPH_FILE, sheet_name='Variables', dtype=str).fillna('')
coding_g = pd.read_excel(GRAPH_FILE, sheet_name='Coding', dtype=str).fillna('')

assigned_terms = pd.read_excel(GRAPH_CT_FILE, sheet_name='AssignedTerms', dtype=str).fillna('')

green_tests = pd.read_excel(TEST_IDENTITY_FILE, sheet_name='Test Codes', dtype=str).fillna('')
domain_meta = pd.read_excel(DOMAIN_META_FILE, sheet_name='Domains', dtype=str).fillna('')

print(f'DSS:           {len(dss_g):>6,} rows')
print(f'BC:            {len(bc_g):>6,} rows')
print(f'Variables:     {len(vars_g):>6,} rows')
print(f'Coding:        {len(coding_g):>6,} rows')
print(f'AssignedTerms: {len(assigned_terms):>6,} rows')
print(f'Test Codes:    {len(green_tests):>6,} rows  (reference, NCIt-enriched)')
print(f'Domains:       {len(domain_meta):>6,} rows  (reference)')


DSS:            1,326 rows
BC:             1,345 rows
Variables:     12,677 rows
Coding:           105 rows
AssignedTerms:  1,170 rows
Test Codes:     5,885 rows  (reference, NCIt-enriched)
Domains:           57 rows  (reference)


## 3. Test_Identity precursor

One row per TESTCD. Discover TESTCDs from `Variables` rows where
`variable_name` ends in `TESTCD` and the row carries an `assigned_term_value`
(the COSMoS author has pinned a specific test-code value on the variable).

Per TESTCD:

- aggregate the set of DSSs that pin it, the set of BCs (via `bc_id` on the
  same Variables row), the set of domains
- attach NCIt enrichment from `AssignedTerms` (graph-native: definition,
  NCI preferred term, used-in-variables count)
- attach the richer NCIt enrichment from `SDTM_Test_Identity.xlsx`
  (synonyms, UMLS_CUI, NCIm_CUI, full domain-membership column)

If the same TESTCD value resolves to more than one NCIt concept across the
graph, the conflict is preserved as semicolon-joined values rather than
silently collapsed.


In [4]:
# TESTCD pins from Variables: variable_name ends in 'TESTCD' and assigned_term_value is non-empty
testcd_pins = vars_g[
    vars_g['variable_name'].str.endswith('TESTCD') &
    (vars_g['assigned_term_value'] != '')
].copy()

# Bring DSS-level domain onto each pin row (Variables already carries ds_id and bc_id)
testcd_pins = testcd_pins.merge(
    dss_g[['ds_id', 'domain']],
    on='ds_id',
    how='left',
).fillna('')

print(f'TESTCD pins: {len(testcd_pins):,} rows  (one per DSS-that-pins-a-TESTCD)')
print(f'Distinct TESTCD values: {testcd_pins["assigned_term_value"].nunique():,}')
print(f'Distinct NCIt concept IDs: {testcd_pins["assigned_term_concept_id"].nunique():,}')


TESTCD pins: 1,120 rows  (one per DSS-that-pins-a-TESTCD)
Distinct TESTCD values: 729
Distinct NCIt concept IDs: 724


In [5]:
# Aggregate to one row per TESTCD value
def _join_unique(s):
    vals = sorted({v for v in s if v})
    return '; '.join(vals)


testcd_agg = testcd_pins.groupby('assigned_term_value', sort=True).agg(
    NCIt_Code=('assigned_term_concept_id', _join_unique),
    DSS_Count=('ds_id', 'nunique'),
    BC_Count=('bc_id', 'nunique'),
    Domains=('domain', _join_unique),
    DS_Codes=('ds_id', _join_unique),
    BC_IDs=('bc_id', _join_unique),
).reset_index().rename(columns={'assigned_term_value': 'TESTCD'})

# Force counts to string for consistent xlsx output
testcd_agg['DSS_Count'] = testcd_agg['DSS_Count'].astype(str)
testcd_agg['BC_Count'] = testcd_agg['BC_Count'].astype(str)

# Flag rows where multiple NCIt codes resolve from the same TESTCD value
testcd_agg['NCIt_Code_Conflict'] = testcd_agg['NCIt_Code'].apply(
    lambda v: 'Yes' if ';' in v else 'No'
)

print(f'Test_Identity precursor: {len(testcd_agg):,} TESTCDs')
print(f'  With NCIt conflict (>1 concept_id for the same TESTCD): {(testcd_agg["NCIt_Code_Conflict"] == "Yes").sum()}')


Test_Identity precursor: 729 TESTCDs
  With NCIt conflict (>1 concept_id for the same TESTCD): 0


In [6]:
# AssignedTerms enrichment (graph-native NCIt resolution).
# Join key: NCIt_Code matches assigned_term_concept_id. Conflict rows
# (multiple NCIt codes per TESTCD) join on the FIRST listed concept;
# the unjoined alternates are still visible in the NCIt_Code column.
at_lookup = assigned_terms.rename(columns={
    'assigned_term_concept_id': '_at_key',
    'used_in_variables_count': 'AssignedTerm_Variable_Uses',
    'term_definition': 'AssignedTerm_Definition',
    'term_nci_preferred_term': 'AssignedTerm_Preferred_Term',
})[['_at_key', 'AssignedTerm_Variable_Uses', 'AssignedTerm_Definition', 'AssignedTerm_Preferred_Term']]

testcd_agg['_at_key'] = testcd_agg['NCIt_Code'].str.split('; ').str[0]
testcd_with_at = testcd_agg.merge(at_lookup, on='_at_key', how='left').fillna('')
testcd_with_at = testcd_with_at.drop(columns=['_at_key'])

resolved_at = (testcd_with_at['AssignedTerm_Preferred_Term'] != '').sum()
print(f'AssignedTerms join: {resolved_at:,} of {len(testcd_with_at):,} TESTCDs resolved by graph CT')


AssignedTerms join: 719 of 729 TESTCDs resolved by graph CT


In [7]:
# Reference enrichment from SDTM_Test_Identity (synonyms, UMLS, NCIm, domain membership).
# Join key: TESTCD value (TEST identity is at TESTCD-grain in the reference).
ref_lookup = green_tests.rename(columns={
    'NCIt_Code': 'Ref_NCIt_Code',
    'NCIt_Preferred_Term': 'Ref_NCIt_Preferred_Term',
    'NCIt_Synonyms': 'Ref_NCIt_Synonyms',
    'NCIt_Definition': 'Ref_NCIt_Definition',
    'UMLS_CUI': 'UMLS_CUI',
    'NCIm_CUI': 'NCIm_CUI',
    'SDTM_Domains': 'Ref_SDTM_Domains',
    'TEST': 'Ref_TEST',
})[['TESTCD', 'Ref_NCIt_Code', 'Ref_NCIt_Preferred_Term', 'Ref_NCIt_Synonyms',
    'Ref_NCIt_Definition', 'UMLS_CUI', 'NCIm_CUI', 'Ref_SDTM_Domains', 'Ref_TEST']]

# Some TESTCDs in the reference resolve to multiple NCIt codes across different
# SDTM domains (TESTCD is namespaced by codelist in CDISC; collision is allowed —
# e.g. COLOR is C64546 in LB/UR and C37927 in OE/PHSPRP). Aggregate to one row
# per TESTCD with semicolon-joined alternates so the precursor stays at TESTCD grain.
def _join_unique_str(s):
    return '; '.join(sorted({v for v in s if v}))

ref_lookup = ref_lookup.groupby('TESTCD', as_index=False).agg({
    'Ref_NCIt_Code': _join_unique_str,
    'Ref_NCIt_Preferred_Term': _join_unique_str,
    'Ref_NCIt_Synonyms': _join_unique_str,
    'Ref_NCIt_Definition': _join_unique_str,
    'UMLS_CUI': _join_unique_str,
    'NCIm_CUI': _join_unique_str,
    'Ref_SDTM_Domains': _join_unique_str,
    'Ref_TEST': _join_unique_str,
})

test_identity = testcd_with_at.merge(ref_lookup, on='TESTCD', how='left').fillna('')

# Flag where graph and reference disagree on the NCIt concept for a TESTCD.
# Both sides may be multi-value (semicolon-joined); disagreement = empty intersection.
def _ncit_disagree(row):
    ref = row['Ref_NCIt_Code']
    graph = row['NCIt_Code']
    if not ref or not graph:
        return ''
    graph_set = {c.strip() for c in graph.split(';') if c.strip()}
    ref_set = {c.strip() for c in ref.split(';') if c.strip()}
    return 'No' if (graph_set & ref_set) else 'Yes'

test_identity['NCIt_Reference_Disagree'] = test_identity.apply(_ncit_disagree, axis=1)

resolved_ref = (test_identity['Ref_NCIt_Preferred_Term'] != '').sum()
print(f'Reference join: {resolved_ref:,} of {len(test_identity):,} TESTCDs found in SDTM_Test_Identity')
print(f'NCIt graph/reference disagreement: {(test_identity["NCIt_Reference_Disagree"] == "Yes").sum()} TESTCDs')

# Final column order for Test_Identity
TEST_IDENTITY_COLS = [
    # COSMoS-side identity (graph)
    'TESTCD',
    'NCIt_Code',
    'NCIt_Code_Conflict',
    'AssignedTerm_Preferred_Term',
    'AssignedTerm_Definition',
    'AssignedTerm_Variable_Uses',
    'DSS_Count',
    'BC_Count',
    'Domains',
    'BC_IDs',
    'DS_Codes',
    # Reference-side identity (SDTM_Test_Identity)
    'Ref_TEST',
    'Ref_NCIt_Code',
    'Ref_NCIt_Preferred_Term',
    'Ref_NCIt_Synonyms',
    'Ref_NCIt_Definition',
    'UMLS_CUI',
    'NCIm_CUI',
    'Ref_SDTM_Domains',
    'NCIt_Reference_Disagree',
]
test_identity = test_identity[TEST_IDENTITY_COLS]
print(f'Test_Identity precursor: {len(test_identity):,} rows x {len(test_identity.columns)} columns')


Reference join: 571 of 729 TESTCDs found in SDTM_Test_Identity
NCIt graph/reference disagreement: 2 TESTCDs
Test_Identity precursor: 729 rows x 20 columns


## 4. Measurement_Specs precursor

One row per DSS. Four-source assembly:

1. **DSS identity** — `DSS` sheet (ds_id, bc_id, domain, source, ds_short_name,
   sdtmig_start_version, sdtmig_end_version, package_date).
2. **BC identity** — `BC` sheet, joined on `bc_id` (bc_short_name, bc_definition,
   bc_categories, bc_hierarchy_path, bc_parent_label, bc_type, result_scales,
   ncit_code as `BC_NCIt_Code`).
3. **External codings** — `Coding` sheet, joined on `bc_id`, pivoted by `system`
   so each coding system gets a column (typically `Coding_LOINC`,
   `Coding_SNOMEDCT`, etc.). Multiple codes per system per BC are
   semicolon-joined.
4. **Variables pins** — lossless pivot. For each `Variables` row in the DSS
   that has a non-empty `assigned_term_value`, the leading domain code is
   stripped from `variable_name` (giving the `--<remainder>` form) and three
   columns are produced per remainder:

   - `<remainder>_value` — the pinned submission value
   - `<remainder>_ncit` — the pinned NCIt concept ID
   - `<remainder>_codelist` — the bound codelist submission value, where present

The remainder set is data-driven, not asserted.


In [8]:
# (1) DSS identity baseline
specs = dss_g.copy()
specs.columns = [c if c == 'ds_id' else f'DSS_{c}' if c in {'bc_id'} else c for c in specs.columns]
# Restore originals — drop the rename we just inserted; we want bc_id intact for the BC join
specs = dss_g.copy()
print(f'(1) DSS identity: {len(specs):,} rows x {len(specs.columns)} columns')


(1) DSS identity: 1,326 rows x 8 columns


In [9]:
# (2) BC identity join
bc_lookup = bc_g.rename(columns={
    'ncit_code': 'bc_ncit_code',
    'bc_short_name': 'bc_short_name',
    'bc_definition': 'bc_definition',
    'bc_synonyms': 'bc_synonyms',
    'bc_categories': 'bc_categories',
    'bc_hierarchy_path': 'bc_hierarchy_path',
    'bc_parent_label': 'bc_parent_label',
    'bc_type': 'bc_type',
    'result_scales': 'result_scales',
})

specs = specs.merge(bc_lookup, on='bc_id', how='left').fillna('')
print(f'(2) After BC join: {len(specs):,} rows x {len(specs.columns)} columns')
print(f'    DSSs with no BC match: {(specs["bc_short_name"] == "").sum()}')


(2) After BC join: 1,326 rows x 17 columns
    DSSs with no BC match: 0


In [10]:
# (3) Coding pivot — one column per coding system, semicolon-joined codes per (BC, system)
coding_pivot = (
    coding_g
    .groupby(['bc_id', 'system'], sort=False)['code']
    .apply(lambda s: '; '.join(sorted({c for c in s if c})))
    .unstack('system', fill_value='')
    .reset_index()
)
# Prefix system columns to keep them distinct in the merged DataFrame
coding_pivot = coding_pivot.rename(
    columns={c: f'Coding_{c}' for c in coding_pivot.columns if c != 'bc_id'}
)
print(f'Coding systems pivoted: {[c for c in coding_pivot.columns if c != "bc_id"]}')

specs = specs.merge(coding_pivot, on='bc_id', how='left').fillna('')
print(f'(3) After Coding pivot: {len(specs):,} rows x {len(specs.columns)} columns')


Coding systems pivoted: ['Coding_http://loinc.org/', 'Coding_https://loinc.org']
(3) After Coding pivot: 1,326 rows x 19 columns


In [11]:
# (4) Variables pins — lossless pivot, leading 2-char domain code stripped from variable_name.
# A pin = a Variables row with non-empty assigned_term_value OR non-empty assigned_term_concept_id.
# Three measures per remainder: _value, _ncit, _codelist.

vars_pinned = vars_g[
    (vars_g['assigned_term_value'] != '') |
    (vars_g['assigned_term_concept_id'] != '')
].copy()

# Bring DSS domain onto each pin row to compute the remainder
vars_pinned = vars_pinned.merge(
    dss_g[['ds_id', 'domain']],
    on='ds_id',
    how='left',
).fillna('')


def _strip_prefix(row):
    name = row['variable_name']
    dom = row['domain']
    if dom and name.startswith(dom):
        return name[len(dom):]
    return name


vars_pinned['var_remainder'] = vars_pinned.apply(_strip_prefix, axis=1)

# Sanity: how many distinct remainders?
remainders = sorted(vars_pinned['var_remainder'].unique())
print(f'Pinned Variables rows: {len(vars_pinned):,}')
print(f'Distinct remainders ({len(remainders)}): {remainders}')


Pinned Variables rows: 4,478
Distinct remainders (42): ['ANMETH', 'ANTXHI', 'ANTXLO', 'ANVLHI', 'ANVLLO', 'BDAGNT', 'CAT', 'DECOD', 'DOSFRM', 'DOSFRQ', 'DOSU', 'ENRF', 'ENRTPT', 'EVAL', 'EVDTYP', 'EVINTX', 'EVLINT', 'GRPID', 'INDC', 'INHERT', 'LLT', 'LLTCD', 'LOC', 'LOINC', 'METHOD', 'OBJ', 'ORRES', 'ORRESU', 'PARM', 'PARMCD', 'PTCD', 'ROUTE', 'SCAT', 'SPEC', 'STRESC', 'STRESU', 'SYMTYP', 'TERM', 'TEST', 'TESTCD', 'TRT', 'TSTDTL']


In [12]:
# Pivot: index ds_id, columns var_remainder, three measures
pin_value = (
    vars_pinned
    .pivot_table(
        index='ds_id', columns='var_remainder',
        values='assigned_term_value',
        aggfunc=lambda s: '; '.join(sorted({v for v in s if v})),
        fill_value='',
    )
    .add_suffix('_value')
    .reset_index()
)

pin_ncit = (
    vars_pinned
    .pivot_table(
        index='ds_id', columns='var_remainder',
        values='assigned_term_concept_id',
        aggfunc=lambda s: '; '.join(sorted({v for v in s if v})),
        fill_value='',
    )
    .add_suffix('_ncit')
    .reset_index()
)

pin_codelist = (
    vars_pinned
    .pivot_table(
        index='ds_id', columns='var_remainder',
        values='codelist_submission_value',
        aggfunc=lambda s: '; '.join(sorted({v for v in s if v})),
        fill_value='',
    )
    .add_suffix('_codelist')
    .reset_index()
)

specs = specs.merge(pin_value, on='ds_id', how='left').fillna('')
specs = specs.merge(pin_ncit, on='ds_id', how='left').fillna('')
specs = specs.merge(pin_codelist, on='ds_id', how='left').fillna('')

print(f'(4) After Variables pin pivot: {len(specs):,} rows x {len(specs.columns)} columns')


(4) After Variables pin pivot: 1,326 rows x 145 columns


In [13]:
# Final column order: identity first, then pins (grouped by remainder, value/ncit/codelist together)
identity_cols = [
    'ds_id', 'bc_id', 'domain', 'source', 'ds_short_name',
    'sdtmig_start_version', 'sdtmig_end_version', 'package_date',
    'bc_ncit_code', 'bc_short_name', 'bc_definition', 'bc_synonyms',
    'bc_categories', 'bc_hierarchy_path', 'bc_parent_label', 'bc_type', 'result_scales',
]
coding_cols = sorted([c for c in specs.columns if c.startswith('Coding_')])

pin_cols = []
for r in remainders:
    for suffix in ('_value', '_ncit', '_codelist'):
        col = f'{r}{suffix}'
        if col in specs.columns:
            pin_cols.append(col)

final_cols = identity_cols + coding_cols + pin_cols
missing = [c for c in final_cols if c not in specs.columns]
if missing:
    raise RuntimeError(f'Missing expected columns: {missing}')

measurement_specs = specs[final_cols].copy()
print(f'Measurement_Specs precursor: {len(measurement_specs):,} rows x {len(measurement_specs.columns)} columns')
print(f'  Identity: {len(identity_cols)}  Coding: {len(coding_cols)}  Pins: {len(pin_cols)}  ({len(remainders)} remainders x up to 3 measures)')


Measurement_Specs precursor: 1,326 rows x 145 columns
  Identity: 17  Coding: 2  Pins: 126  (42 remainders x up to 3 measures)


## 5. Write workbook

Three sheets: `ReadMe`, `Test_Identity`, `Measurement_Specs`. Color convention
follows the repo standard:

- **Green** headers — TESTCD / SDTM-CT-side columns.
- **Yellow** headers — COSMoS-side columns.
- **Grey** headers — keys and aggregation columns.


In [14]:
# Styles
HEADER_FONT = Font(name='Arial', bold=True, size=10, color='FFFFFF')
DATA_FONT = Font(name='Arial', size=10)
WRAP = Alignment(wrap_text=True, vertical='top')

GREEN_HEADER = PatternFill('solid', fgColor='548235')   # TESTCD / SDTM CT side
YELLOW_HEADER = PatternFill('solid', fgColor='FFD700')  # COSMoS side
GREY_HEADER = PatternFill('solid', fgColor='808080')    # keys, aggregation


def write_sheet(ws, df, header_fills, col_widths):
    """Write a DataFrame; header_fills maps col_name -> PatternFill;
    col_widths maps col_name -> width (default 18 if missing)."""
    cols = list(df.columns)
    for ci, name in enumerate(cols, 1):
        cell = ws.cell(row=1, column=ci, value=name)
        cell.font = HEADER_FONT
        cell.fill = header_fills.get(name, GREY_HEADER)
        cell.alignment = WRAP
    for ri, (_, row) in enumerate(df.iterrows(), 2):
        for ci, name in enumerate(cols, 1):
            val = row[name]
            cell = ws.cell(row=ri, column=ci, value=val if val != '' else None)
            cell.font = DATA_FONT
            cell.alignment = WRAP
    for ci, name in enumerate(cols, 1):
        ws.column_dimensions[get_column_letter(ci)].width = col_widths.get(name, 18)
    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = f'A1:{get_column_letter(len(cols))}1'


print('Writer ready.')


Writer ready.


In [15]:
wb = Workbook()

# ── ReadMe sheet ──
ws_rm = wb.active
ws_rm.title = 'ReadMe'

readme_font = Font(name='Arial', size=10)
title_font = Font(name='Arial', size=12, bold=True)
section_font = Font(name='Arial', size=10, bold=True)

readme_lines = [
    ('DSS_View — first consumer-bases view', title_font),
    ('', None),
    ('PROVENANCE', section_font),
    (f'Generated: {datetime.now():%Y-%m-%d %H:%M}', readme_font),
    (f'Notebook: consumer-bases/notebooks/10_dss_view.ipynb', readme_font),
    (f'Inputs:', readme_font),
    (f'  cosmos-graph/interim/COSMoS_Graph.xlsx (DSS, BC, Variables, Coding)', readme_font),
    (f'  cosmos-graph/interim/COSMoS_Graph_CT.xlsx (AssignedTerms)', readme_font),
    (f'  sdtm-test-codes/machine_actionable/SDTM_Test_Identity.xlsx (Test Codes)', readme_font),
    (f'  sdtm-domain-reference/machine_actionable/SDTM_Domain_Metadata.xlsx (Domains)', readme_font),
    ('', None),
    ('SCOPE', section_font),
    ('Graph-wide. Every COSMoS DSS across all 32 domains, not only Findings.', readme_font),
    ('Observation-class scoping (Findings vs Events vs Interventions vs', readme_font),
    ('Special-Purpose) is applied by the consumer track.', readme_font),
    ('', None),
    ('SCOPE DISCIPLINE', section_font),
    ('Joined, denormalised projection of the COSMoS graph plus repo reference', readme_font),
    ('metadata. No interpretive overlays — sub-typing, behavioural classification,', readme_font),
    ('and narrative framing belong to the consumer track. See consumer-bases/README.md.', readme_font),
    ('', None),
    ('SHEETS', section_font),
    ('Test_Identity — one row per TESTCD, derived from Variables TESTCD-pins.', readme_font),
    ('  COSMoS-side: NCIt code (with conflict flag if multiple), aggregation', readme_font),
    ('  counts (DSSs, BCs, domains), AssignedTerms enrichment.', readme_font),
    ('  Reference-side: SDTM_Test_Identity NCIt enrichment (synonyms, UMLS, NCIm,', readme_font),
    ('  full domain membership). NCIt_Reference_Disagree flags graph/reference', readme_font),
    ('  disagreement on the NCIt concept for a TESTCD.', readme_font),
    ('Measurement_Specs — one row per DSS. DSS identity + BC identity + Coding', readme_font),
    ('  pivot (one column per coding system, e.g. Coding_LOINC) + lossless pivot', readme_font),
    ('  of all Variables pins, with the leading 2-char domain code stripped from', readme_font),
    ('  variable_name. Each remainder produces three columns:', readme_font),
    ('    <remainder>_value     — pinned submission value', readme_font),
    ('    <remainder>_ncit      — pinned NCIt concept ID', readme_font),
    ('    <remainder>_codelist  — bound codelist submission value (where present)', readme_font),
    ('  Remainder set is data-driven, not asserted.', readme_font),
    ('', None),
    ('PROVENANCE TRAIL', section_font),
    ('Every column traces back to a graph or reference source. Test_Identity', readme_font),
    ('NCIt enrichment uses graph CT first (AssignedTerms) and reference second', readme_font),
    ('(SDTM_Test_Identity). Conflicts are surfaced as flags, not silently resolved.', readme_font),
    ('', None),
    ('STATUS', section_font),
    ('First consumer-bases view. Not an official CDISC product.', readme_font),
    ('Sources: NCI EVS + COSMoS public exports + repo reference tracks.', readme_font),
]

for ri, (text, font) in enumerate(readme_lines, 1):
    cell = ws_rm.cell(row=ri, column=1, value=text if text else None)
    if font:
        cell.font = font

ws_rm.column_dimensions['A'].width = 100
print(f'ReadMe: {len(readme_lines)} lines')


ReadMe: 45 lines


In [16]:
# ── Test_Identity sheet ──
ws_ti = wb.create_sheet('Test_Identity')

# Header colours: yellow for COSMoS-side, green for reference-side, grey for keys/flags
TI_FILLS = {
    # COSMoS-side (graph-native)
    'TESTCD':                       YELLOW_HEADER,
    'NCIt_Code':                    YELLOW_HEADER,
    'NCIt_Code_Conflict':           GREY_HEADER,
    'AssignedTerm_Preferred_Term':  YELLOW_HEADER,
    'AssignedTerm_Definition':      YELLOW_HEADER,
    'AssignedTerm_Variable_Uses':   GREY_HEADER,
    'DSS_Count':                    GREY_HEADER,
    'BC_Count':                     GREY_HEADER,
    'Domains':                      GREY_HEADER,
    'BC_IDs':                       GREY_HEADER,
    'DS_Codes':                     GREY_HEADER,
    # Reference-side (SDTM_Test_Identity)
    'Ref_TEST':                     GREEN_HEADER,
    'Ref_NCIt_Code':                GREEN_HEADER,
    'Ref_NCIt_Preferred_Term':      GREEN_HEADER,
    'Ref_NCIt_Synonyms':            GREEN_HEADER,
    'Ref_NCIt_Definition':          GREEN_HEADER,
    'UMLS_CUI':                     GREEN_HEADER,
    'NCIm_CUI':                     GREEN_HEADER,
    'Ref_SDTM_Domains':             GREEN_HEADER,
    'NCIt_Reference_Disagree':      GREY_HEADER,
}

TI_WIDTHS = {
    'TESTCD': 14, 'NCIt_Code': 14, 'NCIt_Code_Conflict': 10,
    'AssignedTerm_Preferred_Term': 35, 'AssignedTerm_Definition': 60,
    'AssignedTerm_Variable_Uses': 10,
    'DSS_Count': 8, 'BC_Count': 8, 'Domains': 14, 'BC_IDs': 30, 'DS_Codes': 30,
    'Ref_TEST': 38, 'Ref_NCIt_Code': 14, 'Ref_NCIt_Preferred_Term': 38,
    'Ref_NCIt_Synonyms': 50, 'Ref_NCIt_Definition': 60,
    'UMLS_CUI': 12, 'NCIm_CUI': 12, 'Ref_SDTM_Domains': 18,
    'NCIt_Reference_Disagree': 12,
}

write_sheet(ws_ti, test_identity, TI_FILLS, TI_WIDTHS)
print(f'Test_Identity: {len(test_identity):,} rows x {len(test_identity.columns)} cols')


Test_Identity: 729 rows x 20 cols


In [17]:
# ── Measurement_Specs sheet ──
ws_ms = wb.create_sheet('Measurement_Specs')

# Header colours
MS_FILLS = {
    # DSS identity — yellow (COSMoS-side)
    'ds_id': GREY_HEADER, 'bc_id': GREY_HEADER,
    'domain': YELLOW_HEADER, 'source': YELLOW_HEADER, 'ds_short_name': YELLOW_HEADER,
    'sdtmig_start_version': YELLOW_HEADER, 'sdtmig_end_version': YELLOW_HEADER,
    'package_date': YELLOW_HEADER,
    # BC identity — yellow
    'bc_ncit_code': YELLOW_HEADER, 'bc_short_name': YELLOW_HEADER,
    'bc_definition': YELLOW_HEADER, 'bc_synonyms': YELLOW_HEADER,
    'bc_categories': YELLOW_HEADER, 'bc_hierarchy_path': YELLOW_HEADER,
    'bc_parent_label': YELLOW_HEADER, 'bc_type': YELLOW_HEADER,
    'result_scales': YELLOW_HEADER,
}

MS_WIDTHS = {
    'ds_id': 14, 'bc_id': 14, 'domain': 8, 'source': 16, 'ds_short_name': 38,
    'sdtmig_start_version': 12, 'sdtmig_end_version': 12, 'package_date': 12,
    'bc_ncit_code': 12, 'bc_short_name': 32, 'bc_definition': 50, 'bc_synonyms': 40,
    'bc_categories': 35, 'bc_hierarchy_path': 45, 'bc_parent_label': 30,
    'bc_type': 12, 'result_scales': 18,
}

# Coding columns: yellow
for c in measurement_specs.columns:
    if c.startswith('Coding_'):
        MS_FILLS[c] = YELLOW_HEADER
        MS_WIDTHS[c] = 18

# Pin columns: yellow (assignedTerm side is COSMoS-authored)
for c in measurement_specs.columns:
    if c.endswith('_value') or c.endswith('_ncit') or c.endswith('_codelist'):
        MS_FILLS[c] = YELLOW_HEADER
        MS_WIDTHS[c] = 16

write_sheet(ws_ms, measurement_specs, MS_FILLS, MS_WIDTHS)
print(f'Measurement_Specs: {len(measurement_specs):,} rows x {len(measurement_specs.columns)} cols')


Measurement_Specs: 1,326 rows x 145 cols


In [18]:
wb.save(OUTPUT_FILE)
print(f'\nWritten: {OUTPUT_FILE}')
print(f'File size: {OUTPUT_FILE.stat().st_size / 1024:.0f} KB')



Written: /sessions/quirky-epic-cray/mnt/cdisc-for-ai/consumer-bases/interim/DSS_View.xlsx
File size: 747 KB


## 6. Summary


In [19]:
print('=== DSS_View summary ===')
print(f'Output: {OUTPUT_FILE.relative_to(REPO_ROOT)}')
print()
print(f'Test_Identity:    {len(test_identity):>5,} rows  x {len(test_identity.columns):>3} cols')
print(f'  NCIt conflicts (multi-concept TESTCDs): {(test_identity["NCIt_Code_Conflict"] == "Yes").sum()}')
print(f'  Reference disagreements:                {(test_identity["NCIt_Reference_Disagree"] == "Yes").sum()}')
print(f'  Resolved by AssignedTerms:              {(test_identity["AssignedTerm_Preferred_Term"] != "").sum():,}')
print(f'  Resolved by SDTM_Test_Identity:         {(test_identity["Ref_NCIt_Preferred_Term"] != "").sum():,}')
print()
print(f'Measurement_Specs: {len(measurement_specs):>5,} rows  x {len(measurement_specs.columns):>3} cols')
print(f'  Domains:    {sorted(measurement_specs["domain"].unique())}')
print(f'  Pinned remainders ({len(remainders)}): {remainders}')
coding_systems = [c.replace("Coding_", "") for c in measurement_specs.columns if c.startswith("Coding_")]
print(f'  Coding systems: {coding_systems}')


=== DSS_View summary ===
Output: consumer-bases/interim/DSS_View.xlsx

Test_Identity:      729 rows  x  20 cols
  NCIt conflicts (multi-concept TESTCDs): 0
  Reference disagreements:                2
  Resolved by AssignedTerms:              719
  Resolved by SDTM_Test_Identity:         571

Measurement_Specs: 1,326 rows  x 145 cols
  Domains:    ['AE', 'BE', 'CM', 'DD', 'DM', 'DS', 'EC', 'EG', 'EX', 'FA', 'FT', 'GF', 'IE', 'IS', 'LB', 'MB', 'MH', 'MI', 'MK', 'PR', 'QS', 'RE', 'RP', 'RS', 'SC', 'SR', 'SU', 'TR', 'TS', 'TU', 'UR', 'VS']
  Pinned remainders (42): ['ANMETH', 'ANTXHI', 'ANTXLO', 'ANVLHI', 'ANVLLO', 'BDAGNT', 'CAT', 'DECOD', 'DOSFRM', 'DOSFRQ', 'DOSU', 'ENRF', 'ENRTPT', 'EVAL', 'EVDTYP', 'EVINTX', 'EVLINT', 'GRPID', 'INDC', 'INHERT', 'LLT', 'LLTCD', 'LOC', 'LOINC', 'METHOD', 'OBJ', 'ORRES', 'ORRESU', 'PARM', 'PARMCD', 'PTCD', 'ROUTE', 'SCAT', 'SPEC', 'STRESC', 'STRESU', 'SYMTYP', 'TERM', 'TEST', 'TESTCD', 'TRT', 'TSTDTL']
  Coding systems: ['http://loinc.org/', 'https://loi